# 06 · 评测集：调参用哪份，最后考哪份，放在哪

## 这一章在整个项目里的位置

```
00 总览 → 01 存档旧项目 ✅ → 02 新项目骨架 ✅ → 03 自动检查 ✅ → 04 GitHub 与 CI ✅ → 05 准备考题 ✅ → [06 评测集 ✅] → 07 判分 ✅ → 08 MLflow 评测 ✅ → 09 第一个基线 ✅ → 10 第一个工具 ✅ → 11 第二个工具 ✅ → 12 起：更多工具、agent …
```

> **这一章的代码在合并请求 #4 里。** 下面的命令重复运行安全：题号清单已经冻结，再运行只会显示 `unchanged`。

## 背景：一万例考题，不能"想用哪些用哪些"

05 号有了一万多例考题，也分好了开发集和留出集。真正用起来，还缺三样：

1. **日常调参要一份小的。** agent 每做一道题都要调好几次大模型，6,014 例全跑一遍又慢又花钱，不可能每改一次提示词就跑一遍
2. **"考的是哪些题"要记下来。** 旧项目的评测结果里，有 6 例、25 例、60 例、150 例好几种规模（`otovce_v2/eval/` 里的 `ablation_n6`、`ablation_n25`、`ablation_main_n60`……），每次临时抽题，题不一样，结果之间没法比
3. **数据放在哪、从哪来，要有统一的规矩。** 05 号的版本号和指纹直接写在代码里。以后加 RareBench、RareArena，每个都这么写就乱了

**公司里的做法**：
- 评测集分几份，每份管一件事：日常调参、阶段性确认、自动检查、最终考试
- **每份评测集是一张冻结的题号清单，进 git**；数据本身不进 git
- 数据分层存放：原始的永远不改，整理后的由代码生成
- 所有数据源写在一张清单里

## 这一章做的 5 件事

| # | 做什么 | 用的技术 | 解决什么问题 |
|---|---|---|---|
| 1 | 数据源清单 | `evals/datasets.toml` | 版本、地址、指纹、许可只写在一个地方 |
| 2 | 数据分两层存放 | `data/raw` → `data/processed` | 原始数据永远不改；整理结果随时能重建 |
| 3 | 抽日常调参用的 200 例 | 每篇论文 1 例 + 固定种子 | 覆盖面广，而且每次都是同一批 |
| 4 | 冻结 4 张题号清单 | `evals/splits/*.tsv`，进 git | "考的是哪些题"永远说得清 |
| 5 | 给清单定规矩，自动检查 | 契约测试，CI 里跑 | 清单被改坏，CI 变红 |

**花费：0。**

---

## 准备

运行下面这一格。它定义一个叫 `run` 的小工具，用来执行命令并显示结果。**不用看代码。**

> 运行 `uv` 的命令时，有时会先出现几行 `Building` / `Installed`：那是 uv 发现代码有变化，顺手更新了一下环境，可以忽略。

In [ ]:
import os, subprocess
from pathlib import Path
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
os.chdir(ROOT)
os.environ.update(NO_COLOR="1", GH_PAGER="cat", GIT_PAGER="cat")
def run(cmd, check=True):
    print(f"$ {cmd}")
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    print(r.stdout + r.stderr)
    if check and r.returncode != 0:
        raise RuntimeError(f"failed: {cmd}")
print("项目目录:", ROOT)

---

## 第 1 步 · 数据源清单

### 在干什么
把每个评测数据集"从哪来、哪个版本、怎么核对、能不能用"写进一张清单：[`evals/datasets.toml`](../../evals/datasets.toml)。

### 背景
05 号的时候，版本号和指纹直接写在读数据的代码里。现在只有一个数据集还好；以后加 RareBench、RareArena，这些信息就散落在各处，想查"我们到底用了哪些数据、哪个版本"，得把代码翻一遍。

### 用了什么技术，为什么
清单是 TOML 格式（`pyproject.toml` 也是），人能直接读。每个数据集一段：

| 字段 | 例子 | 作用 |
|---|---|---|
| `version` | `0.1.27` | 版本写死 |
| `url` | GitHub 上的发布地址 | 从哪下载 |
| `sha256` | `d0c7…` | 指纹，下载后核对 |
| `license` | `BSD-3-Clause` | 能不能用、能不能转发 |
| `role` | 主评测集 | 在项目里干什么 |

**代码只从这里读**，别处不再写版本号。这和你在 qafi-ml-platform 03b 里的 `config/sources.toml` 是同一个做法。

### 公司里什么情况怎么做
- **加一个数据集**：清单里加一段，再写一个把它转成统一病例格式的读取器，别的代码不用改
- **官方出了新版本**：改 `version` 和 `sha256`，新数据进新文件夹，旧的不动
- **有授权限制的数据**（医院数据、MIMIC）：清单里写明"不能转发"；数据只放在被批准的地方，绝不进 git

### 执行

In [ ]:
run("cat evals/datasets.toml")

### 结果怎么看
一段 `[phenopacket-store]`，就是上面表里那几个字段。最上面几行注释写着规矩：代码只从这里读。

### 面试怎么说
> "Every evaluation dataset is declared in one version-controlled registry with its version, URL, checksum and license; code reads it, so adding a dataset is one entry plus a parser."

---

## 第 2 步 · 数据分两层存放

### 在干什么
原始下载放在 `data/raw`，整理好的病例放在 `data/processed`。之后所有代码都只读 `processed`。

### 背景
原始文件是 10,377 个层层嵌套的 JSON，每次用都要重新解析一遍。更要紧的是：如果直接在原始文件上改，改坏了就回不去了。

### 用了什么技术，为什么
就是你在 qafi-data-platform 里学过的分层：

```
data/
├── raw/phenopacket-store/0.1.27/
│   └── all_phenopackets.zip        原样下载，永远不改                         ← bronze
└── processed/phenopacket-store/0.1.27/
    ├── cases.jsonl                 一行一个病例，统一格式                     ← silver
    └── build.json                  出厂记录：用的哪个版本、哪个指纹、多少病例
```

- `raredx data build`：读 raw，转成 05 号的统一病例格式，写进 processed
- processed 由代码生成，**删了也没关系**，一条命令重建
- `build.json` 是"出厂记录"。以后任何一个结果，都能追溯到它是从哪一份原始数据来的
- 文件格式是 JSONL：一行一个病例，打开就能看

两层都在 `data/` 里，都不进 git。

### 公司里什么情况怎么做
- **数据大了**（几个 GB 以上）：processed 层改用 Parquet 这类格式，放在云上的存储（S3）里，而不是本地。道理一样
- **整理规则改了**：改代码，重新 build；raw 不动

### 执行

In [ ]:
run("uv run raredx data build")
run("cat data/processed/phenopacket-store/0.1.27/build.json")

### 结果怎么看
- `ok  data/processed/phenopacket-store/0.1.27/cases.jsonl  (10374 cases, skipped {'no_phenotype': 3})`
- `build.json` 里：`source`、`version`、`raw_sha256` 和数据源清单里的一致；`cases` 是 10374

### 面试怎么说
> "Raw downloads are immutable; a build step turns them into one processed file with a provenance record, and everything downstream reads only the processed layer."

---

## 第 3 步 · 抽日常调参用的 200 例

### 在干什么
从开发集（6,014 例）里抽 200 例，作为每次改动之后都跑一遍的"小考"。

### 背景
- **太多不行**：每改一次提示词就跑 6,014 例，要几个小时，还要花不少钱
- **太少也不行**：几例、几十例，看到的差别多半是运气

200 例是常见的折中，但要知道它的精度：准确率在 50% 左右时，200 例的误差范围大约是**正负 7 个百分点**。两个版本差 5 个点以内，200 例分不出谁好，要到 6,014 例上确认（误差约正负 1.3 个点）。

### 用了什么技术，为什么
两条规则：
- **每篇论文最多 1 例**：开发集里最大的一篇论文有 462 例。随便抽的话，会抽进十几例来自同一篇论文、同一家人的病例，等于把同一道题考很多遍
- **固定种子**：抽样用的随机数从一个固定的数（`20261008`）开始，所以不管谁运行、运行多少次，抽出来的都是同一批

代码在 [`src/raredx/evals/splits.py`](../../src/raredx/evals/splits.py)。

> **一个要知道的偏差**：每篇论文只取 1 例，等于让只有一两个病人的小论文占了更大比重，而小论文往往描述得更细：dev-200 每例症状数的中位数是 9，整个开发集是 7。所以 dev-200 比整体**稍微好做一点**。用它比较两个版本谁更好没有问题，但最终成绩不能用它报。

### 公司里什么情况怎么做
- **日常**：每改一次跑 dev-200；大改之后，在开发集全量上确认
- **抽样的偏差要写下来**：报告里说清楚"这个数是在哪份题上测的、这份题有什么特点"

### 执行
把 4 份评测集各自的情况列出来比一比：

In [ ]:
import statistics
from collections import Counter
from raredx.data import cases as case_store, sources
from raredx.evals import splits

source = sources.load_sources(ROOT / "evals/datasets.toml")["phenopacket-store"]
by_id = {c.id: c for c in case_store.read(sources.processed_dir(source, ROOT / "data"))}

print(f"{'manifest':<9}{'cases':>7}{'papers':>8}{'diseases':>10}{'genes':>7}{'median_terms':>14}{'max_per_paper':>15}")
for name in splits.NAMES:
    cs = [by_id[i] for i, _ in splits.read(ROOT / "evals/splits" / f"{name}.tsv")]
    print(f"{name:<9}{len(cs):>7}{len({c.pmid for c in cs}):>8}{len({c.disease_id for c in cs}):>10}"
          f"{len({c.gene for c in cs if c.gene}):>7}{statistics.median(len(c.hpo_ids) for c in cs):>14}"
          f"{max(Counter(c.pmid for c in cs).values()):>15}")

### 结果怎么看

| 清单 | 病例 | 论文 | 病 | 基因 | 症状数中位数 | 同一篇论文最多几例 |
|---|---|---|---|---|---|---|
| design | 6,014 | 893 | 522 | 478 | 7 | 462 |
| dev-200 | 200 | 200 | 170 | 165 | 9 | 1 |
| ci-20 | 20 | 20 | 19 | 19 | 10 | 1 |
| heldout | 4,360 | 840 | 495 | 449 | 8 | 80 |

dev-200 的 200 例来自 200 篇不同的论文，覆盖 170 种病、165 个基因。如果不限制每篇 1 例，同一篇论文最多可能占几十例。

### 面试怎么说
> "Day-to-day iteration runs on a 200-case dev set sampled one case per source paper with a fixed seed. It catches large effects quickly; anything inside its ±7-point noise is confirmed on the full design split."

---

## 第 4 步 · 冻结 4 张题号清单

### 在干什么
把 4 份评测集各写成一张题号清单，放进 `evals/splits/`，提交进 git。

### 背景
如果"开发集"只是代码里临时算出来的，那么代码一改、数据版本一换，同样叫"开发集"的题就可能变了，而你不会发现。旧项目就是这样，每次评测临时抽题。

### 用了什么技术，为什么

| 清单 | 内容 | 什么时候用 |
|---|---|---|
| `design.tsv` | 开发集全部，6,014 例 | 大改之后确认 |
| `dev-200.tsv` | 200 例，每篇论文 1 例 | 每改一次 |
| `ci-20.tsv` | 从 dev-200 里再抽 20 例 | 自动检查（以后配上录好的模型回答） |
| `heldout.tsv` | 留出集全部，4,360 例 | 设计定稿后，只跑一次 |

每张清单是一个两列的表：病例编号、来源论文。最大的一张 250 KB，放进 git 没有负担。

**冻结**：清单生成以后，`raredx eval splits` 再运行，只核对、不覆盖。如果数据变了、清单会跟着变，它直接拒绝，提示你另建一个名字不同的版本。题目一旦定下来就不改，这叫**冻结的评测集**（frozen eval set）。

### 公司里什么情况怎么做
- **真要换题**：起一个新名字（比如 `dev-200-v2`），旧的留着，提交说明里写清楚为什么换。以前的结果仍然能和以前的题对上
- **清单的改动走合并请求审查**，和代码一样

### 执行

In [ ]:
run("uv run raredx eval splits")
run("head -4 evals/splits/dev-200.tsv")

### 结果怎么看
- 4 行 `unchanged`：清单已经冻结，再运行只核对、不改
- 清单的前几行：表头 `case_id  pmid`，然后每行一个病例

### 面试怎么说
> "Each eval set is a frozen manifest of case IDs committed to git, so 'the dev set' means the same cases for everyone, forever; a change is a new, named version."

---

## 第 5 步 · 给清单定规矩，自动检查

### 在干什么
把清单必须遵守的规矩写下来，让机器每次都检查。

### 背景
清单是普通的文本文件，手能改。有人不小心把一例留出集的题加进了开发集：看上去只是多了一行，最后考试却漏了答案。

### 用了什么技术，为什么
`raredx eval check` 检查这些规矩：

| 规矩 | 需要数据吗 |
|---|---|
| 同一张清单里没有重复的病例 | 不需要 |
| 一个病例、一篇论文，不能同时在开发集和留出集 | 不需要 |
| dev-200 都来自开发集，ci-20 都来自 dev-200 | 不需要 |
| dev-200 正好 200 例、每篇论文 1 例；ci-20 正好 20 例 | 不需要 |
| 清单里的每个病例都真实存在；开发集、留出集清单和数据完全对得上 | 需要 |

不需要数据的那几条，同时写成了测试，**每个合并请求 GitHub 都会跑**：CI 上没有数据，也照样能检查清单。

这和你在 qafi-data-platform 里给 dbt 写的测试是同一件事：把"数据应该满足的条件"写成能自动运行的检查，叫**契约测试**（data contract test）。

### 公司里什么情况怎么做
- **每次评测开始前先检查**，不通过就不开跑
- **检查失败**：先查是谁、为什么改了清单，而不是改规矩让它通过

### 执行
先检查真正的清单。再演示：把清单复制到一个临时文件夹，故意把一例留出集的病例加进 dev-200，看检查能不能抓到（临时文件夹用完就删）。最后跑这一章的测试：

In [ ]:
import shutil, tempfile

run("uv run raredx eval check")

tmp = Path(tempfile.mkdtemp())
for f in (ROOT / "evals/splits").glob("*.tsv"):
    shutil.copy(f, tmp)
heldout_row = (tmp / "heldout.tsv").read_text().splitlines()[1]
with open(tmp / "dev-200.tsv", "a") as f:
    f.write(heldout_row + "\n")
run(f"uv run raredx eval check --splits-dir {tmp}", check=False)
shutil.rmtree(tmp)

run("uv run pytest tests/evals -vv")   # -vv：列出每条测试的名字

### 结果怎么看
- 真正的清单：`ok  evals/splits: design, dev-200, ci-20, heldout pass every rule`
- 改坏的清单：两条 `problem`：`dev-200 contains cases that are not in design` 和 `dev-200 has 201 cases, expected 200`
- 测试：6 条 `PASSED`。最后一条 `test_committed_manifests_keep_every_rule`，就是 CI 里检查真正清单的那一条

### 面试怎么说
> "The manifests have contract tests — no case or paper on both sides of the split, dev inside design, exact sizes — that run in CI without the data, plus a data-backed check before every evaluation run."

---

## 本章小结

| 步骤 | 技术 | 公司里的作用 |
|---|---|---|
| 1 数据源清单 | `evals/datasets.toml` | 用了哪些数据、哪个版本，只有一个答案 |
| 2 两层存放 | `data/raw` → `data/processed` + 出厂记录 | 原始数据不改；结果能追溯到数据 |
| 3 日常调参的 200 例 | 每篇 1 例 + 固定种子 | 快，覆盖面广，每次同一批 |
| 4 冻结清单 | `evals/splits/*.tsv` 进 git | "考哪些题"永远说得清 |
| 5 契约测试 | `raredx eval check` + CI | 清单被改坏，马上发现 |

**一句话**：考题分成了四份，每份是一张冻结在 git 里的题号清单；数据分层存放，来源写在一张清单里。从这一章起，"在开发集上是多少分"有了唯一的意思。

## 面试题

1. **How big should a dev set be?**
   > "Big enough that the differences you care about exceed its noise. At 50% accuracy, 200 cases give roughly ±7 points, so I use it for fast iteration and confirm smaller effects on the full design split."

2. **Why freeze evaluation sets?**
   > "Otherwise numbers from different weeks aren't comparable, and nobody can tell whether the model changed or the test set did. A frozen manifest pins the exact cases; a change is a new, named version."

3. **How do you keep test data from leaking into development?**
   > "Split by the unit that carries shared information — here the source paper — commit the split as a manifest, and enforce disjointness with tests in CI. The held-out set is run once, at the end."

**下一章 07 · 判分**：agent 给出的病名，和病例的确诊答案对不对得上，自动打分。